In [1]:
import pandas as pd
import numpy as np
df = pd.read_excel('data.xlsx')
print(df.shape)
df.head()


(100, 71)


,年龄,性别,脑出血前mRS评分,高血压病史,卒中病史,糖尿病史,房颤史,冠心病史,吸烟史,饮酒史,...,NCCT_original_firstorder_Mean,NCCT_original_firstorder_Median,NCCT_original_firstorder_Minimum,NCCT_original_firstorder_Range,NCCT_original_firstorder_RobustMeanAbsoluteDeviation,NCCT_original_firstorder_RootMeanSquared,NCCT_original_firstorder_Skewness,NCCT_original_firstorder_Uniformity,NCCT_original_firstorder_Variance,Class
0,43,女,0,0,0,0,0,0,0,0,...,60.479072,62.363129,25.625693,68.636402,6.426142,61.421799,-0.464143,0.115781,114.919202,0
1,58,男,0,1,0,0,0,0,0,0,...,63.060397,63.564973,25.287045,70.502845,4.876413,63.676979,-0.196186,0.141840,78.143978,0
2,78,男,0,1,0,0,0,0,0,0,...,63.057433,63.207367,25.162122,86.779325,5.970173,64.096201,0.158409,0.137676,132.083050,1
3,70,男,2,1,1,0,0,0,0,0,...,48.338894,48.073538,8.273075,82.476919,7.699185,50.029409,-0.005145,0.109191,166.293121,0
4,51,男,0,0,0,0,0,0,0,0,...,49.344928,50.737956,16.756305,61.928524,6.387522,50.518385,-0.384037,0.101547,117.185249,1


In [2]:
X = df.drop(columns='Class')
y = df.Class
X['性别'] = X['性别'].map({'男':1, '女':0})
bp = X['血压'].str.split('/', expand=True)
X['收缩压'] = pd.to_numeric(bp[0], errors='raise')
X['舒张压'] = pd.to_numeric(bp[1], errors='raise')
X = X.drop(columns=['血压'])
X.shape, y.shape

((100, 71), (100,))

<font size=5>模型专属重要性（embedded methods）</font>

依赖于算法内部机制，例如随机森林的 Gini 重要性或 XGBoost 的增益（gain）等。

这类重要性对同一模型往往有效，但不一定能迁移到其他模型上，否则可能导致新的模型性能下降

<font size=5>模型无关重要性（model‑agnostic）</font>

如置换重要性（permutation importance）直接测量打乱特征后对任意模型性能的影响，更能反映特征真实贡献，并可无缝应用于后续任何模型

SHAP 值基于博弈论的 Shapley 值，量化每个特征在所有可能组合中的边际贡献，既能给出全局排序，也能解释单个样本的预测

因此，尚未锁定最终预测模型时，依赖单一模型专属的重要性可能带来偏差—一旦改用其他模型，先前筛选的特征不一定仍然最优

# RandomForest

## Gini importance (MDI)

在决策树中，Gini 不纯度用于衡量一个节点中样本的“混杂”程度

若所有样本都属于同一类别，则 Gini 为 0，越接近 0 表示节点越“纯”

每次用某特征进行分裂时，会计算分裂前后子节点的不纯度变化（ΔGini）

对同一棵树，统计该特征在所有节点上造成的 ΔGini 之和，就得到该特征对这棵树整体纯度提升的贡献

在随机森林中，对所有树的贡献求和，再做归一化处理，就得到每个特征的 Gini 重要性（也叫 Mean Decrease in Impurity，MDI）

In [3]:
# 基于多次随机森林训练计算特征重要性
# 无需先标准化

from sklearn.ensemble import RandomForestClassifier 

num_iterations = 10
feature_importance_list = []

for iteration in range(num_iterations):
    rf_classifier = RandomForestClassifier(n_estimators=100)
    rf_classifier.fit(X, y)

    # rf_classifier.feature_importances_ 给出每个特征的 Gini 重要性（节点纯度提升之和）
    feature_importance = rf_classifier.feature_importances_
    # 归一化到 [0,1]
    min_score, max_score = feature_importance.min(), feature_importance.max()
    normalized = [(s - min_score)/(max_score - min_score) for s in feature_importance]

    feature_importance_list.append(normalized)

In [4]:
# 汇总并选出最重要特征

average_imp = np.mean(feature_importance_list, axis=0)
top_13_indices = np.argsort(average_imp)[-20:]
top_13_features = X.columns[top_13_indices]
print(top_13_features)

Index(['original_shape_Sphericity', 'original_shape_SurfaceArea', 'ED_volume',
       'NCCT_original_firstorder_Uniformity',
       'NCCT_original_firstorder_Entropy',
       'NCCT_original_firstorder_RobustMeanAbsoluteDeviation',
       'NCCT_original_firstorder_MeanAbsoluteDeviation',
       'NCCT_original_firstorder_Maximum',
       'NCCT_original_firstorder_10Percentile',
       'NCCT_original_firstorder_Kurtosis', 'original_shape_VoxelVolume',
       'original_shape_MajorAxisLength', 'original_shape_MeshVolume',
       'NCCT_original_firstorder_Range', 'NCCT_original_firstorder_Skewness',
       'original_shape_Flatness', 'NCCT_original_firstorder_Variance',
       'original_shape_Maximum2DDiameterRow',
       'original_shape_Maximum3DDiameter', 'NCCT_original_firstorder_Minimum'],
      dtype='object')


MDI（Gini 重要性）的偏差

随机森林的 MDI（Mean Decrease in Impurity）在高基数（high‑cardinality）或连续变量上往往会高估重要性，因为这类特征更容易在决策树分裂时产生较大不纯度下降

对于高度相关（multicollinear）的特征组，MDI 可能将重要性分散到整个组，导致单个特征评分偏低

## Out-of-Bag Permutation Importance

置换重要性直接度量“打乱该特征后模型分数下降多少”，与特征的分布或类型无关，更能反映对最终预测性能的实际贡献

可用于任何已训练模型（非线性、黑箱模型都可），不依赖内部节点不纯度计算，因而更具通用性

置换重要性原理

基准评分：先用原始数据计算模型在验证集（或 OOB 样本）上的性能指标，如准确率、AUC、MSE 等

单特征置换：对某个特征列随机打乱其值（保持其他特征不变），再次计算模型性能

重要性得分：该特征的重要性定义为“基准分数 – 打乱后分数”，分数下降越多，说明模型越依赖该特征

重复与平均：可多次重复置换并取平均，以降低随机波动。

In [5]:
# 使用 s

from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

# 1. 划分训练/测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# 2. 训练随机森林并启用 OOB
rf = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)
rf.fit(X_train, y_train)

# 3. 基于测试集计算置换重要性（也可传 X_train, y_train + 设置 n_repeats 多次重复）
result = permutation_importance(rf, X_test, y_test,
                                n_repeats=10,
                                random_state=42,
                                scoring='accuracy')
# 4. 排序并输出
importances = result.importances_mean
indices = importances.argsort()[::-1]
top_features = X.columns[indices]

## 真正关心的不是绝对大小，而是相对排序
print("置换重要性排序：", list(zip(top_features, importances[indices]*100)))

置换重要性排序： [('舒张压', np.float64(0.0)), ('HM_PCA_L_Ratio', np.float64(0.0)), ('HM_MCA_R_Ratio', np.float64(0.0)), ('HM_PCA_R_Ratio', np.float64(0.0)), ('HM_Pons_Medulla_R_Ratio', np.float64(0.0)), ('HM_Cerebellum_R_Ratio', np.float64(0.0)), ('HM_ACA_L_Ratio', np.float64(0.0)), ('HM_MCA_L_Ratio', np.float64(0.0)), ('HM_Pons_Medulla_L_Ratio', np.float64(0.0)), ('营养神经', np.float64(0.0)), ('HM_Cerebellum_L_Ratio', np.float64(0.0)), ('ED_volume', np.float64(0.0)), ('ED_ACA_R_Ratio', np.float64(0.0)), ('ED_MCA_R_Ratio', np.float64(0.0)), ('ED_PCA_R_Ratio', np.float64(0.0)), ('ED_Pons_Medulla_R_Ratio', np.float64(0.0)), ('HM_ACA_R_Ratio', np.float64(0.0)), ('止吐护胃', np.float64(0.0)), ('ED_ACA_L_Ratio', np.float64(0.0)), ('冠心病史', np.float64(0.0)), ('性别', np.float64(0.0)), ('脑出血前mRS评分', np.float64(0.0)), ('高血压病史', np.float64(0.0)), ('卒中病史', np.float64(0.0)), ('糖尿病史', np.float64(0.0)), ('房颤史', np.float64(0.0)), ('吸烟史', np.float64(0.0)), ('镇静、镇痛治疗', np.float64(0.0)), ('饮酒史', np.float64(0.0)), ('发病到首次影

permutation_importance 会自动重复打乱、计算模型分数差值并返回 importances_mean 和 importances_std

In [6]:
# 手动在 OOB 上计算置换重要性

import numpy as np
from sklearn.utils import shuffle

# 训练启用 OOB
rf = RandomForestClassifier(n_estimators=100, oob_score=True, bootstrap=True, random_state=0)
rf.fit(X, y)

# 原始 OOB 准确率
base_oob_score = rf.oob_score_

oob_importances = {}
for feature in X.columns:
    # 复制一份训练集，将该特征列打乱
    X_permuted = X.copy()
    X_permuted[feature] = shuffle(X_permuted[feature], random_state=0)
    # 重计算每棵树的 OOB 预测（sklearn 不直接支持，这里示意）
    # 可调用 sklearn.inspection.permutation_importance 指定 OOB，但手动则需自定义 OOB 预测
    permuted_oob_score = rf.oob_score_  # 简化：实际需重算
    oob_importances[feature] = base_oob_score - permuted_oob_score

# 按降序排序
sorted(oob_importances.items(), key=lambda x: x[1], reverse=True)

[('年龄', 0.0),
 ('性别', 0.0),
 ('脑出血前mRS评分', 0.0),
 ('高血压病史', 0.0),
 ('卒中病史', 0.0),
 ('糖尿病史', 0.0),
 ('房颤史', 0.0),
 ('冠心病史', 0.0),
 ('吸烟史', 0.0),
 ('饮酒史', 0.0),
 ('发病到首次影像检查时间间隔', 0.0),
 ('脑室引流', 0.0),
 ('止血治疗', 0.0),
 ('降颅压治疗', 0.0),
 ('降压治疗', 0.0),
 ('镇静、镇痛治疗', 0.0),
 ('止吐护胃', 0.0),
 ('营养神经', 0.0),
 ('HM_ACA_R_Ratio', 0.0),
 ('HM_MCA_R_Ratio', 0.0),
 ('HM_PCA_R_Ratio', 0.0),
 ('HM_Pons_Medulla_R_Ratio', 0.0),
 ('HM_Cerebellum_R_Ratio', 0.0),
 ('HM_ACA_L_Ratio', 0.0),
 ('HM_MCA_L_Ratio', 0.0),
 ('HM_PCA_L_Ratio', 0.0),
 ('HM_Pons_Medulla_L_Ratio', 0.0),
 ('HM_Cerebellum_L_Ratio', 0.0),
 ('ED_volume', 0.0),
 ('ED_ACA_R_Ratio', 0.0),
 ('ED_MCA_R_Ratio', 0.0),
 ('ED_PCA_R_Ratio', 0.0),
 ('ED_Pons_Medulla_R_Ratio', 0.0),
 ('ED_Cerebellum_R_Ratio', 0.0),
 ('ED_ACA_L_Ratio', 0.0),
 ('ED_MCA_L_Ratio', 0.0),
 ('ED_PCA_L_Ratio', 0.0),
 ('ED_Pons_Medulla_L_Ratio', 0.0),
 ('ED_Cerebellum_L_Ratio', 0.0),
 ('original_shape_Flatness', 0.0),
 ('original_shape_LeastAxisLength', 0.0),
 ('original_shape_

# XGBoost

使用 XGBoost 进行特征筛选，核心思路是基于模型训练后输出的“特征重要性”（feature importance）来保留对预测最有贡献的变量。

XGBoost 提供三种内置重要性度量——“weight”（分裂次数）、“gain”（分裂增益）和“cover”（样本覆盖

我们可以借助它们配合阈值过滤（如 SelectFromModel）、递归特征消除（RFE）或 SHAP 值来完成自动化特征选择。

weight：特征在所有树中被用作分裂节点的次数

gain：使用该特征分裂带来的目标函数（如对数损失）平均减少量，是最常用的度量

cover：分裂时所覆盖样本的平均数量，反映特征影响的样本规模

在 Python API 中，可通过 model.get_score(importance_type=…) 或 xgboost.plot_importance(model, importance_type=…) 获取并可视化这些指标

## 基于阈值筛选

In [7]:
# 基于阈值的简单筛选：SelectFromModel

from xgboost import XGBClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split

# 1. 划分数据
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# 2. 训练 XGBoost
model = XGBClassifier(eval_metric='logloss')
model.fit(X_train, y_train)

# 3. 基于平均增益（gain）自动设阈值筛选

# 创建选择器（不使用 prefit）
selector = SelectFromModel(model, threshold='mean', prefit=False)

# 拟合选择器并记录列名
selector.fit(X_train, y_train)

# 获取保留的特征列名
selected_columns = X_train.columns[selector.get_support()].tolist()
print("保留列名：", selected_columns)

保留列名： ['发病到首次影像检查时间间隔', '止血治疗', 'HM_ACA_R_Ratio', 'HM_MCA_R_Ratio', 'HM_ACA_L_Ratio', 'HM_PCA_L_Ratio', 'HM_Pons_Medulla_L_Ratio', 'ED_volume', 'ED_MCA_R_Ratio', 'ED_PCA_R_Ratio', 'ED_ACA_L_Ratio', 'original_shape_MajorAxisLength', 'original_shape_Maximum3DDiameter', 'original_shape_MeshVolume', 'original_shape_MinorAxisLength', 'original_shape_SurfaceVolumeRatio', 'NCCT_original_firstorder_10Percentile', 'NCCT_original_firstorder_Kurtosis', 'NCCT_original_firstorder_Mean', 'NCCT_original_firstorder_Variance', '收缩压', '舒张压']


Scikit‑learn 提供了 SelectFromModel 元转换器，能根据指定阈值保留重要性 ≥ 阈值的特征

对 XGBoost 来说，它会读取 feature_importances_（对应“gain”）或 get_score 输出。

## RFE

In [8]:
# 递归特征消除（RFE）

from sklearn.feature_selection import RFE

# 保留前 15 个特征
rfe = RFE(estimator=model, n_features_to_select=15, step=1)
rfe.fit(X_train, y_train)

# RFE 支持 .support_ 和 .ranking_
selected_features = X.columns[rfe.support_].to_list()
print("RFE 选出的特征：", selected_features)


RFE 选出的特征： ['发病到首次影像检查时间间隔', '止血治疗', 'HM_MCA_R_Ratio', 'HM_PCA_R_Ratio', 'ED_volume', 'ED_MCA_R_Ratio', 'ED_PCA_R_Ratio', 'ED_ACA_L_Ratio', 'original_shape_MajorAxisLength', 'original_shape_Maximum3DDiameter', 'original_shape_MinorAxisLength', 'NCCT_original_firstorder_Kurtosis', 'NCCT_original_firstorder_Mean', 'NCCT_original_firstorder_Range', '舒张压']


RFE 每次训练后剔除最不重要的特征，直至达到指定数量。可与 XGBoost 结合使用

## SHAP

在博弈论中，Shapley 值用于衡量每个参与者在合作游戏中对整体收益的贡献。将这一概念应用于机器学习中，SHAP 值评估每个特征在所有可能的特征组合中对预测结果的边际贡献。这种方法确保了特征重要性的公平分配，具有良好的理论基础

SHAP 值的优势

- 局部解释：SHAP 值可以解释单个预测结果中各个特征的影响，帮助我们理解模型在特定样本上的决策依据。

- 全局解释：通过聚合所有样本的 SHAP 值，可以评估特征在整个数据集中的重要性，提供全局视角。

- 一致性：SHAP 值满足一致性属性，即如果一个特征对模型的贡献增加，其 SHAP 值不会减少。

- 模型无关性：SHAP 方法适用于多种模型，包括树模型（如 XGBoost、LightGBM）和神经网络等。

In [9]:
import shap

explainer = shap.Explainer(model)
shap_values = explainer(X_train)

# 平均绝对 SHAP 值
shap_importance = np.abs(shap_values.values).mean(axis=0)
shap_idx = np.argsort(shap_importance)[::-1][:15]
top_shap_features = X.columns[shap_idx]
print("SHAP Top-15 特征：", list(top_shap_features))


SHAP Top-15 特征： ['NCCT_original_firstorder_Kurtosis', 'ED_MCA_R_Ratio', '舒张压', 'HM_PCA_L_Ratio', 'ED_PCA_R_Ratio', 'original_shape_Maximum3DDiameter', 'NCCT_original_firstorder_90Percentile', 'HM_PCA_R_Ratio', 'ED_ACA_R_Ratio', 'NCCT_original_firstorder_Energy', 'original_shape_Flatness', 'NCCT_original_firstorder_Range', 'NCCT_original_firstorder_Minimum', 'original_shape_MinorAxisLength', 'HM_ACA_L_Ratio']


| 方法             | 计算复杂度                            | 随机波动                  | 处理相关特征                                              |
| -------------- | -------------------------------- | --------------------- | --------------------------------------------------- |
| 置换重要性          | O(n\_features × model\_eval)     | 中等；可增加 `n_repeats` 平滑 | 易受高度相关特征影响，需要先去相关 |
| SHAP（TreeSHAP） | O(n\_trees × depth × n\_samples) | 低；确定性算法（TreeSHAP）     | 自动考虑特征交互与相关性分配            |



Flora 等人在气象领域对比了多种解释方法，发现去除高度相关特征后，置换重要性在“忠实度”（fidelity）上优于 SHAP，但在有相关特征时 SHAP 更稳定

Qlik Cloud 文档指出，若关注单行预测或特征交互，应优先 SHAP；若仅需全局重要性且追求效率，置换重要性足矣

Lanchu Huong 实验表明，对同一模型，SHAP 平均重要性与置换重要性总体排序相似，但在排序细节上常有差异，SHAP 能更好捕捉低频但关键的交互效应